# Lab 10 - Encrypted Firmware CTF

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

The goal for this lab is to attack an encrypted firmware update mechanism. See `background.md` for additional information. The SAM4S's datasheet may also be helpful for this lab. Again, instruction will be minimal.

You will have to:

1. Figure out how to send firmware to the target
2. Capture synchronized traces
3. Figure out a leakage model and perform a CPA attack
4. Decrypt the firmware

You may want to use the attack workflow from the previous video as well. TODO: do this in video but not lab

Hints will be provided for the steps. Good luck!

## Prerequisites

Hold up! Make sure that you've completed all of the previous labs before trying to tackle this one!

In [ ]:
import chipwhisperer as cw
scope = cw.scope()
target = cw.target(scope)
scope.default_setup()

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "bootloader.hex")

<details><summary>Part A Hint</summary>

There are three parts to the firmware communication. The first is a 2 byte length field. Then, there's a data field. Finally, there's a 32-bit CRC.

The SAM4S has a CRC peripheral that is used to speed up this calculation. The following details of the CRC are important:

1. The polynomial. The CRC peripheral has 2 possible polynomials, so you'll need to figure out which one the target is using
2. The initial value of the CRC
3. Bit ordering

The CRC module is available to help you with the CRC calculation:

```python
import crc

config = crc.Configuration(width=32, polynomial=????, init_value=????, final_xor_value=0x00, reverse_input=????)
calculator = crc.Calculator(config)

cs = calculator.checksum(data)
```

To get into the firmware update mechanism, you need to set TIO3 low. TIO2 functions as a reset pin when the target is booted and held low for 5 seconds.
</details>

<details><summary>Part B Hint</summary>

Capturing synchronized traces is very similar to the previous lab. You'll need to send everything except the last byte of the CRC, then resynchronize with SAD.

Here's the SAD code from the previous lab:

```python
import numpy as np
def sad_resync(trace, ref, ref_range):
    pattern = ref[ref_range[0]:ref_range[1]]
    off, diff = sad(trace, pattern)
    #print(off)
    actual_offset = ref_range[0] - off
    new = np.zeros(len(ref))
    try:
        if actual_offset > 0:
            new[actual_offset:] = trace[:-actual_offset]
        elif actual_offset < 0:
            new[:actual_offset] = trace[-actual_offset:]
        else:
            new[:] = trace[:]
    except:
        print(off, diff, len(new), actual_offset)
        print(new)
        print(trace)
        raise
    return new
    
def sad(trace, pattern):
    tlen = len(trace)
    plen = len(pattern)
    best_diff = 100000000
    best_offset = 0
    for i in range(tlen-plen):
        diff = np.sum(np.abs(trace[i:i+plen] - pattern[:]))
        #print(diff)
        if diff < best_diff:
            best_offset = i
            best_diff = diff
    
    return best_offset, best_diff
```

Pay attention to the after effects for sending firmware. You may need to do something between firmware captures to get the target to decrypt more firmware.
</details>

<details><summary>Part C Hint</summary>

From our traces, we can see that this algorithm looks a lot like our previous AES128 traces, so that's probably the encryption that the target is using. There are two main ways this firmware could
be decrypted. The first is a chaining cipher mode like CBC, in which case we should be able to run a normal CPA attack. The next is a counter based mode like CTR, in which case we'll need
to do additional work for a CPA attack.

A simple way we can distinguish the two is by simply running a CPA attack. We're attacking a decryption here, so we'll need to change our leakage model to reflect that. Take a look at the following block diagram for AES decryption
to see if you can figure out how to construct your leakage model.

You may find the following imports helpful:

```python
from chipwhisperer.analyzer.leakage_models import inv_sbox, sbox_output, hw_table
```

To help you, here's the source code for the `sbox_output` leakage model function (also available by running `sbox_output??` in a code block):

```python
def sbox_output(pt, ct, subkey):
    if pt is None:
        raise ValueError("This leakage model requires plaintext")
    rtn = np.zeros((len(pt[0]), 256), dtype=np.uint8)
    for kguess in range(256):
        rtn[:, kguess] = hw_table[sbox[pt[subkey] ^ kguess]]
    return rtn
```
</details>

<details><summary>Part D Hint</summary>

You should have recovered an AES key in the previous part (though it won't be highlighted because we don't know the key), strongly indicating that this is AES128-CBC. 
The final thing you need to do to complete the attack is to actually decrypt the firmware.

There's a few roadblocks:

1. AES derives a different key for each round from the original key. This means that we recovered the 10th round key and will need to reverse the expansion algorithm to get the actual AES key.
2. We don't know the IV

ChipWhisperer has a utility to change a round key to a different round:

```python
from chipwhisperer.analyzer.utils import key_schedule_rounds
new_key = key_schedule_rounds(key, i_round, o_round)
```

The second issue is trickier to solve. The best method of attacking the IV is to do a Differential Power Analysis (DPA) attack. This is similar to what we did in Lab 5, where we separated our power traces into
high and low Hamming weight values. DPA attacks weren't covered in this course, but feel free to give it a try if you'd like.

You can use PyCryptoDome to decrypt:

```python
from Crypto.Cipher import AES
cipher = AES.new(bytearray(key), AES.MODE_CBC, iv=bytearray(IV))
```
</details>

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>